
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/master/13_ml_intro/13_ml_intro.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 13 — Introduction to Machine Learning

**Learning Objectives:** What ML is, supervised learning, the ML workflow, train/test splits, cross-validation, evaluation metrics

**Estimated time:** 60–90 minutes

---

## 13.1 What Is Machine Learning?

**The traditional programming paradigm:**
You write explicit rules. The computer follows them. If the input does not fit your rules, it fails. Example: writing rules for spam detection means listing every spam phrase manually — impossible to keep up with spammers.

**The machine learning paradigm:**
Instead of writing rules, you provide examples (labelled data) and let the algorithm figure out the rules itself. The algorithm finds patterns in the examples that generalise to new, unseen data.

```
Traditional: Rules + Data     -> Output
ML:          Data + Output    -> Rules (called a "model")
```

**Three types of machine learning:**

| Type | How it learns | Examples |
|------|--------------|---------|
| **Supervised** | From labelled examples (input + correct answer) | Spam detection, house price prediction, cancer diagnosis |
| **Unsupervised** | From unlabelled data, finds hidden structure | Customer segmentation, anomaly detection, dimensionality reduction |
| **Reinforcement** | By trial and error with rewards/penalties | Game playing (AlphaGo), robotics, trading algorithms |

**This course focuses on supervised learning** — the most widely used type in industry.

**Supervised learning has two forms:**
- **Classification** — predict a category (spam/not spam, cancer/benign, which digit 0-9)
- **Regression** — predict a continuous number (house price, temperature, stock return)

## 13.2 The ML Workflow

**Every supervised ML project follows the same steps:**

```
1. Define the problem    What are we predicting? What counts as success?
2. Collect data          Gather labelled examples
3. Explore data (EDA)    Understand distributions, spot issues
4. Preprocess            Clean, encode, scale features
5. Split                 Hold out a test set BEFORE any processing
6. Train                 Fit a model on training data
7. Evaluate              Measure performance on held-out test data
8. Iterate               Improve: try different models, tune hyperparameters
9. Deploy                Put the model into production
```

**The most important rule:** Hold out your test data FIRST, before any exploration or preprocessing. If you peek at test data during development — even accidentally — your evaluation is no longer honest. This is called **data leakage** and it is the most common mistake beginners make.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Step 1: Load data
iris = load_iris(as_frame=True)
X = iris.data      # features: 4 measurements per flower
y = iris.target    # labels: 0, 1, 2 (three species)

print("Dataset shape:", X.shape)
print("Feature names:", list(X.columns))
print("Classes:", iris.target_names.tolist())
print("
Class distribution (balanced dataset):")
print(y.value_counts().sort_index())
print("
First few rows:")
print(X.head())

In [ ]:
# Step 2: Exploratory Data Analysis
# Visualise how the four features separate the three classes
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for ax, feature in zip(axes.flat, X.columns):
    for label in y.unique():
        ax.hist(X[feature][y == label], bins=15, alpha=0.6,
                label=iris.target_names[label])
    ax.set_title(feature)
    ax.legend(fontsize=8)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.suptitle("Feature distributions by class", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("Observation: petal length and petal width separate the classes most clearly.")

In [ ]:
# Step 3: Train/test split — do this BEFORE any preprocessing
# stratify=y ensures class proportions are the same in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 80% train, 20% test
    random_state=42,   # reproducible split
    stratify=y         # preserve class balance in both sets
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"
Train class distribution:
{y_train.value_counts().sort_index()}")
print(f"
Test class distribution:
{y_test.value_counts().sort_index()}")

In [ ]:
# Step 4: Preprocessing — scale features
# StandardScaler transforms each feature to have mean=0 and std=1
# This matters for distance-based algorithms (SVM, KNN) and regularised models

# CRITICAL: fit only on training data, then transform both train and test
# Fitting on test data would leak test information into training — data leakage!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean/std from train, then scale
X_test_scaled  = scaler.transform(X_test)         # use the SAME mean/std to scale test

print("Before scaling (first row):")
print(X_train.iloc[0].values.round(3))

print("
After scaling (first row):")
print(X_train_scaled[0].round(3))
print("(mean of each feature in training set is now ~0, std is ~1)")

In [ ]:
# Step 5: Train and evaluate
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Confusion matrix — rows are actual, columns are predicted
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 13.3 Cross-Validation

**The problem with a single train/test split:**
A single split can be lucky or unlucky. If your test set happens to contain the easiest examples, your accuracy will be over-estimated. If it contains the hardest, it will be under-estimated.

**What cross-validation does:**
Instead of one split, k-fold cross-validation makes k splits. In 5-fold CV:
1. Split data into 5 equal parts (folds)
2. Train on folds 1,2,3,4 — test on fold 5. Record score.
3. Train on folds 1,2,3,5 — test on fold 4. Record score.
4. Repeat until each fold has been the test set once.
5. Average the 5 scores.

**Result:** Every data point gets used for both training AND testing. The averaged score is a much more reliable estimate of real-world performance.

**When to use:** Always use cross-validation when selecting between models or tuning hyperparameters. Use a final held-out test set only for the very last evaluation.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# 5-fold cross-validation — 5 train/test splits, averaged
cv_scores = cross_val_score(
    LogisticRegression(max_iter=200),
    X, y,
    cv=5,
    scoring="accuracy"
)

print("Individual fold scores:", cv_scores.round(3))
print(f"Mean accuracy:  {cv_scores.mean():.3f}")
print(f"Std deviation:  {cv_scores.std():.3f}")
print(f"95% confidence: {cv_scores.mean():.3f} +/- {2*cv_scores.std():.3f}")
print()
print("The std deviation tells you how variable the performance is across splits.")
print("A high std means the model is sensitive to which data it trains on.")

## 13.4 Evaluation Metrics

**Why accuracy alone is not enough:**
Imagine a fraud detection model. Only 1% of transactions are fraudulent. A model that always predicts "not fraud" would have 99% accuracy — but it catches zero fraud. Accuracy is misleading when classes are imbalanced.

**Classification metrics:**

| Metric | Formula | When to use |
|--------|---------|------------|
| **Accuracy** | (TP+TN) / Total | Only when classes are balanced |
| **Precision** | TP / (TP+FP) | When false positives are costly (e.g. spam filter — don't mark real emails as spam) |
| **Recall** | TP / (TP+FN) | When false negatives are costly (e.g. cancer screening — don't miss real cases) |
| **F1 Score** | 2 * P*R / (P+R) | When you need to balance precision and recall |
| **ROC-AUC** | Area under ROC curve | Overall discrimination ability, good for imbalanced classes |

**Regression metrics:**

| Metric | What it measures | Notes |
|--------|----------------|-------|
| **MAE** | Average absolute error | Easy to interpret, robust to outliers |
| **RMSE** | Root mean squared error | Same units as target, penalises large errors more |
| **R²** | Proportion of variance explained | 1 = perfect, 0 = no better than mean, negative = worse than mean |

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score)
import numpy as np

# Simulate a fraud detection scenario
# 97% of transactions are genuine (0), 3% are fraud (1)
np.random.seed(42)
n = 1000
y_true = np.random.choice([0, 1], n, p=[0.97, 0.03])

# "Dumb" model: always predicts not fraud
y_pred_dumb = np.zeros(n, dtype=int)

# "Smart" model: actually catches some fraud
y_pred_smart = y_true.copy()
# Simulate some mistakes
mistake_idx = np.random.choice(np.where(y_true==1)[0], size=5, replace=False)
y_pred_smart[mistake_idx] = 0

for name, preds in [("Dumb (always 0)", y_pred_dumb), ("Smart model", y_pred_smart)]:
    print(f"
{name}:")
    print(f"  Accuracy:  {accuracy_score(y_true, preds):.3f}")
    print(f"  Precision: {precision_score(y_true, preds, zero_division=0):.3f}")
    print(f"  Recall:    {recall_score(y_true, preds):.3f}")
    print(f"  F1 Score:  {f1_score(y_true, preds, zero_division=0):.3f}")

print()
print("The dumb model has 97% accuracy but 0% recall — it catches ZERO fraud cases!")
print("This is why accuracy alone is misleading on imbalanced datasets.")

---

## Key Takeaways

- ML learns patterns from data instead of following hand-coded rules
- **Supervised learning:** labelled data -> model -> predictions
- **Split data FIRST** before any exploration or preprocessing — prevent data leakage
- Scale features AFTER splitting — fit the scaler on train data only
- **Cross-validation** gives a more reliable performance estimate than a single split
- Choose your metric based on the business problem — accuracy alone is often misleading

## Next: [14 — ML Algorithms](../14_ml_algorithms/14_ml_algorithms.ipynb)
